In [3]:
import os
import re
import glob
import pandas as pd
import numpy as np
import xarray as xr

# ── 配置路径 ──────────────────────────────────────────
POP_MARK_PATH = r"E:\wyy\SPDB_database\data\raw\1\pop_mark.nc"
GEO_CLF_CSV = r"C:\Users\dell\OneDrive\file\csv\geo_clf_updated.csv"


def load_geo_classification_masks(csv_path):
    """
    读取 geo_clf_grid.csv，并构建 continent / development 的二维掩膜。
    返回：
        continent_masks_raw: dict[str, xr.DataArray]
        development_masks_raw: dict[str, xr.DataArray]
    """
    df = pd.read_csv(csv_path)

    required_cols = {'lon', 'lat', 'continent', 'development'}
    if not required_cols.issubset(df.columns):
        raise ValueError(f"{csv_path} 必须包含列: {required_cols}")

    valid_continents = ['Africa', 'Asia', 'Europe', 'Latin America', 'North America']
    valid_development = ['Developing regions', 'Developed regions']

    grid = df[['lat', 'lon']].drop_duplicates()
    lat_vals = np.sort(grid['lat'].unique())
    lon_vals = np.sort(grid['lon'].unique())

    continent_masks_raw = {}
    for cont in valid_continents:
        tmp = df[['lat', 'lon', 'continent']].copy()
        tmp['mask'] = (tmp['continent'] == cont).astype(np.int8)
        pivot = tmp.pivot(index='lat', columns='lon', values='mask')
        pivot = pivot.reindex(index=lat_vals, columns=lon_vals).fillna(0).astype(np.int8)

        continent_masks_raw[cont] = xr.DataArray(
            pivot.values,
            coords={'lat': lat_vals, 'lon': lon_vals},
            dims=('lat', 'lon'),
            name=cont
        )

    development_masks_raw = {}
    for dev in valid_development:
        tmp = df[['lat', 'lon', 'development']].copy()
        tmp['mask'] = (tmp['development'] == dev).astype(np.int8)
        pivot = tmp.pivot(index='lat', columns='lon', values='mask')
        pivot = pivot.reindex(index=lat_vals, columns=lon_vals).fillna(0).astype(np.int8)

        development_masks_raw[dev] = xr.DataArray(
            pivot.values,
            coords={'lat': lat_vals, 'lon': lon_vals},
            dims=('lat', 'lon'),
            name=dev
        )

    return continent_masks_raw, development_masks_raw


def weighted_mean_timeseries(data_3d, mask_2d, weights_2d):
    """
    对 3D 数组 data_3d(year, lat, lon) 在 mask_2d 范围内计算逐年面积加权均值。
    返回 shape=(year,)
    """
    valid = mask_2d[None, :, :] & np.isfinite(data_3d)
    w = np.where(valid, weights_2d[None, :, :], 0.0)
    d = np.where(valid, data_3d, 0.0)

    denom = w.sum(axis=(1, 2))
    numer = (d * w).sum(axis=(1, 2))

    out = np.full(data_3d.shape[0], np.nan, dtype=float)
    good = denom > 0
    out[good] = numer[good] / denom[good]
    return out


def process_nc_files(input_dir, output_dir, target_vars=None):
    """
    批量读取 NetCDF 文件，计算：
      - 全球均值（面积加权）
      - 有人区均值（面积加权）
      - 无人区均值（面积加权）
      - 各大洲均值（面积加权）
      - 发达/发展中地区均值（面积加权）

    参数：
        input_dir : str
            NetCDF 文件夹路径
        output_dir : str
            输出 CSV 文件夹路径
        target_vars : list[str] or None
            指定只处理哪些变量；
            若为 None，则默认处理文件中的所有 data_vars
    """

    ds_pop = xr.open_dataset(POP_MARK_PATH)
    pop_mark_raw = ds_pop['population']
    ds_pop.close()

    continent_masks_raw, development_masks_raw = load_geo_classification_masks(GEO_CLF_CSV)

    results = []
    nc_files = glob.glob(os.path.join(input_dir, '*.nc'))

    continent_col_map = {
        'Africa': 'africa',
        'Asia': 'asia',
        'Europe': 'europe',
        'Latin America': 'latin_america',
        'North America': 'north_america'
    }
    development_col_map = {
        'Developing regions': 'developing',
        'Developed regions': 'developed'
    }

    for nc_path in nc_files:
        fname = os.path.basename(nc_path)

        if fname.startswith('sw_forecast'):
            seed_match = re.search(r'sw_forecast_(\d+)', fname)
            seed = seed_match.group(1) if seed_match else 'unknown'
        elif fname.startswith('lr_forecast'):
            seed_match = re.search(r'lr_forecast_(\d+_\d+)', fname)
            seed = seed_match.group(1) if seed_match else 'unknown'
        else:
            seed = 'unknown'

        ds = xr.open_dataset(nc_path)

        if 'lat' not in ds.coords or 'lon' not in ds.coords:
            ds.close()
            raise ValueError(f"{fname} 不包含坐标 lat/lon")
        if 'year' not in ds.coords:
            ds.close()
            raise ValueError(f"{fname} 不包含坐标 year")

        all_vars = list(ds.data_vars)

        if target_vars is None:
            vars_to_process = all_vars
        else:
            vars_to_process = [var for var in target_vars if var in all_vars]

        if not vars_to_process:
            print(f"⚠️ 文件 {fname} 中没有匹配的变量，跳过")
            ds.close()
            continue

        years = ds['year'].values
        lat = ds['lat'].values
        lon = ds['lon'].values

        pop_mark = pop_mark_raw.interp(
            lat=ds['lat'], lon=ds['lon'], method='nearest'
        )
        inhabited_mask = (pop_mark == 1).values
        uninhabited_mask = (pop_mark == 0).values
        global_mask = np.ones((len(lat), len(lon)), dtype=bool)

        continent_masks = {}
        for cont, da in continent_masks_raw.items():
            continent_masks[cont] = (
                da.interp(lat=ds['lat'], lon=ds['lon'], method='nearest').values == 1
            )

        development_masks = {}
        for dev, da in development_masks_raw.items():
            development_masks[dev] = (
                da.interp(lat=ds['lat'], lon=ds['lon'], method='nearest').values == 1
            )

        lat_weights = np.cos(np.deg2rad(lat)).astype(float)
        weights_2d = np.broadcast_to(lat_weights[:, None], (len(lat), len(lon)))

        region_masks = {
            'g': global_mask,
            'ih': inhabited_mask,
            'uih': uninhabited_mask,
        }

        for cont, short_name in continent_col_map.items():
            region_masks[short_name] = continent_masks[cont]

        for dev, short_name in development_col_map.items():
            region_masks[short_name] = development_masks[dev]

        for var in vars_to_process:
            data = ds[var]

            if not all(dim in data.dims for dim in ['year', 'lat', 'lon']):
                print(f"⚠️ {fname} 的变量 {var} 不包含 year/lat/lon，跳过")
                continue

            data = data.transpose('year', 'lat', 'lon')
            data_3d = data.values

            stats = {}
            for region_name, mask_2d in region_masks.items():
                stats[f'{region_name}_mean'] = weighted_mean_timeseries(
                    data_3d, mask_2d, weights_2d
                )

            for i, y in enumerate(years):
                row = {
                    'pfas': var,
                    'seed': seed,
                    'year': int(y),

                    'g_mean': stats['g_mean'][i],
                    'ih_mean': stats['ih_mean'][i],
                    'uih_mean': stats['uih_mean'][i],

                    'africa_mean': stats['africa_mean'][i],
                    'asia_mean': stats['asia_mean'][i],
                    'europe_mean': stats['europe_mean'][i],
                    'latin_america_mean': stats['latin_america_mean'][i],
                    'north_america_mean': stats['north_america_mean'][i],

                    'developing_mean': stats['developing_mean'][i],
                    'developed_mean': stats['developed_mean'][i],
                }
                results.append(row)

        ds.close()

    df = pd.DataFrame(results)

    os.makedirs(output_dir, exist_ok=True)
    outfile = os.path.join(output_dir, 'global_stats_short.csv')
    df.to_csv(outfile, index=False, encoding='utf-8-sig')
    print(f"✅ 已保存结果到: {outfile}")

In [4]:
    
    
if __name__ == '__main__':
    process_nc_files(
        input_dir=r"F:\User_file\wyy\SPDB\part3_forecast\sw\output_2020\raw_final",
        output_dir=r"F:\User_file\wyy\SPDB\part3_forecast\sw\output_2020",
        target_vars=['lc_value', 'sc_value', 'value'],   # 例如 ['PFOA', 'PFOS']
    )

✅ 已保存结果到: F:\User_file\wyy\SPDB\part3_forecast\sw\output_2020\global_stats_short.csv


In [5]:
    
    
if __name__ == '__main__':
    process_nc_files(
        input_dir=r"F:\User_file\wyy\SPDB\part3_forecast\lr\output_2020\raw_final",
        output_dir=r"F:\User_file\wyy\SPDB\part3_forecast\lr\output_2020",
        target_vars=['lc_value', 'sc_value', 'value'],   # 例如 ['PFOA', 'PFOS']
    )

✅ 已保存结果到: F:\User_file\wyy\SPDB\part3_forecast\lr\output_2020\global_stats_short.csv


In [8]:
import os
import numpy as np
import pandas as pd


def process_global_stats(file_path):
    """
    读取 global_stats.csv，按不同区域分别汇总，
    每个区域输出一个 summary 文件。

    汇总方式（按 pfas, year 分组）：
      - 对 mean 列：
          mean_mean, mean_std, mean_sem
          mean_ci95_low, mean_ci95_high   # 基于 SEM 的均值置信区间
          mean_q2_5, mean_q20, mean_q50, mean_q80, mean_q97_5  # 基于 seed 分布的经验分位数
      - 对 median 列：
          median_mean
          median_std
          median_median
          median_MAD
          median_q2_5, median_q20, median_q50, median_q80, median_q97_5
      - n_seeds
    """
    df = pd.read_csv(file_path)

    # 各区域的列映射：(输出文件后缀, mean列名, median列名)
    regions = [
        ("g", "g_mean"),
        ("ih", "ih_mean"),
        ("uih", "uih_mean"),
        ("africa", "africa_mean"),
        ("asia", "asia_mean"),
        ("europe", "europe_mean"),
        ("latin_america", "latin_america_mean"),
        ("north_america", "north_america_mean"),
        ("developing", "developing_mean"),
        ("developed", "developed_mean"),
    ]

    def safe_std(x):
        return np.std(x, ddof=1) if len(x) > 1 else np.nan

    def safe_sem(x):
        if len(x) > 1:
            std = np.std(x, ddof=1)
            return std / np.sqrt(len(x))
        return np.nan

    def safe_mad(x):
        if len(x) == 0:
            return np.nan
        med = np.median(x)
        return np.median(np.abs(x - med))

    def agg_func(mean_col):
        def _agg(group):
            mean_vals = group[mean_col].dropna().values

            # mean 统计
            if len(mean_vals) > 0:
                mean_mean = np.mean(mean_vals)
                mean_std = safe_std(mean_vals)
                mean_sem = safe_sem(mean_vals)
                mean_q2_5 = np.percentile(mean_vals, 2.5)
                mean_q20 = np.percentile(mean_vals, 20)
                mean_q50 = np.percentile(mean_vals, 50)
                mean_q80 = np.percentile(mean_vals, 80)
                mean_q97_5 = np.percentile(mean_vals, 97.5)
            else:
                mean_mean = np.nan
                mean_std = np.nan
                mean_sem = np.nan
                mean_q2_5 = np.nan
                mean_q20 = np.nan
                mean_q50 = np.nan
                mean_q80 = np.nan
                mean_q97_5 = np.nan


            return pd.Series({
                "mean_mean": mean_mean,
                "mean_std": mean_std,
                "mean_sem": mean_sem,
                "mean_q2_5": mean_q2_5,
                "mean_q20": mean_q20,
                "mean_q50": mean_q50,
                "mean_q80": mean_q80,
                "mean_q97_5": mean_q97_5,
                "n_mean_seeds": len(mean_vals),
            })
        return _agg

    base_dir = os.path.dirname(file_path)

    for suffix, mean_col in regions:
        required = {"pfas", "seed", "year", mean_col}
        if not required.issubset(df.columns):
            missing = required - set(df.columns)
            raise ValueError(f"区域 {suffix} 缺少必要列: {missing}")

        result_df = (
            df.groupby(["pfas", "year"], sort=False)
              .apply(agg_func(mean_col))
              .reset_index()
        )

        out_path = os.path.join(base_dir, f"global_stats_short_{suffix}.csv")
        result_df.to_csv(out_path, index=False, float_format="%.6f")
        print(f"✅ 已保存: {out_path}")

In [9]:
# 使用示例：
paths = [
    r"F:\User_file\wyy\SPDB\part3_forecast\sw\output_2020\global_stats_short.csv",
]
for p in paths:
    process_global_stats(p)

✅ 已保存: F:\User_file\wyy\SPDB\part3_forecast\sw\output_2020\global_stats_short_g.csv
✅ 已保存: F:\User_file\wyy\SPDB\part3_forecast\sw\output_2020\global_stats_short_ih.csv
✅ 已保存: F:\User_file\wyy\SPDB\part3_forecast\sw\output_2020\global_stats_short_uih.csv
✅ 已保存: F:\User_file\wyy\SPDB\part3_forecast\sw\output_2020\global_stats_short_africa.csv
✅ 已保存: F:\User_file\wyy\SPDB\part3_forecast\sw\output_2020\global_stats_short_asia.csv
✅ 已保存: F:\User_file\wyy\SPDB\part3_forecast\sw\output_2020\global_stats_short_europe.csv
✅ 已保存: F:\User_file\wyy\SPDB\part3_forecast\sw\output_2020\global_stats_short_latin_america.csv
✅ 已保存: F:\User_file\wyy\SPDB\part3_forecast\sw\output_2020\global_stats_short_north_america.csv
✅ 已保存: F:\User_file\wyy\SPDB\part3_forecast\sw\output_2020\global_stats_short_developing.csv
✅ 已保存: F:\User_file\wyy\SPDB\part3_forecast\sw\output_2020\global_stats_short_developed.csv


In [10]:
# 使用示例：
paths = [
    r"F:\User_file\wyy\SPDB\part3_forecast\lr\output_2020\global_stats_short.csv",
]
for p in paths:
    process_global_stats(p)

✅ 已保存: F:\User_file\wyy\SPDB\part3_forecast\lr\output_2020\global_stats_short_g.csv
✅ 已保存: F:\User_file\wyy\SPDB\part3_forecast\lr\output_2020\global_stats_short_ih.csv
✅ 已保存: F:\User_file\wyy\SPDB\part3_forecast\lr\output_2020\global_stats_short_uih.csv
✅ 已保存: F:\User_file\wyy\SPDB\part3_forecast\lr\output_2020\global_stats_short_africa.csv
✅ 已保存: F:\User_file\wyy\SPDB\part3_forecast\lr\output_2020\global_stats_short_asia.csv
✅ 已保存: F:\User_file\wyy\SPDB\part3_forecast\lr\output_2020\global_stats_short_europe.csv
✅ 已保存: F:\User_file\wyy\SPDB\part3_forecast\lr\output_2020\global_stats_short_latin_america.csv
✅ 已保存: F:\User_file\wyy\SPDB\part3_forecast\lr\output_2020\global_stats_short_north_america.csv
✅ 已保存: F:\User_file\wyy\SPDB\part3_forecast\lr\output_2020\global_stats_short_developing.csv
✅ 已保存: F:\User_file\wyy\SPDB\part3_forecast\lr\output_2020\global_stats_short_developed.csv


### 空间std

In [1]:
import xarray as xr
import numpy as np

file_path = r"F:\User_file\wyy\SPDB\part3_forecast\sw\output_2020\sw_value_year_mean.nc"

ds = xr.open_dataset(file_path)
data = ds["sw_year_mean"]

print("变量维度:", data.dims)

# 面积权重
weights = np.cos(np.deg2rad(ds["lat"]))

# 面积加权均值
mean_w = data.weighted(weights).mean(dim=("lat", "lon"), skipna=True)

# 面积加权标准差
var_w = ((data - mean_w) ** 2).weighted(weights).mean(dim=("lat", "lon"), skipna=True)
std_w = np.sqrt(var_w)

print(f"面积加权均值: {float(mean_w.values):.6f}")
print(f"面积加权标准差: {float(std_w.values):.6f}")

变量维度: ('lat', 'lon')
面积加权均值: 8.158564
面积加权标准差: 48.539636


In [2]:
import xarray as xr
import numpy as np

file_path = r"F:\User_file\wyy\SPDB\part3_forecast\lr\output_2020\lr_value_year_mean.nc"

ds = xr.open_dataset(file_path)
data = ds["lr_year_mean"]

print("变量维度:", data.dims)

# 面积权重
weights = np.cos(np.deg2rad(ds["lat"]))

# 面积加权均值
mean_w = data.weighted(weights).mean(dim=("lat", "lon"), skipna=True)

# 面积加权标准差
var_w = ((data - mean_w) ** 2).weighted(weights).mean(dim=("lat", "lon"), skipna=True)
std_w = np.sqrt(var_w)

print(f"面积加权均值: {float(mean_w.values):.6f}")
print(f"面积加权标准差: {float(std_w.values):.6f}")

变量维度: ('lat', 'lon')
面积加权均值: 3.557317
面积加权标准差: 2.206621


In [1]:
import xarray as xr
import numpy as np


def weighted_quantile_1d(values, weights, quantiles):
    """
    计算一维加权分位数

    Parameters
    ----------
    values : 1D array
        数据
    weights : 1D array
        对应权重
    quantiles : float or array-like
        分位点，范围 [0, 1]

    Returns
    -------
    np.ndarray
        对应分位数结果
    """
    values = np.asarray(values)
    weights = np.asarray(weights)
    quantiles = np.atleast_1d(quantiles)

    # 去除 NaN
    mask = np.isfinite(values) & np.isfinite(weights) & (weights > 0)
    values = values[mask]
    weights = weights[mask]

    if values.size == 0:
        return np.full(len(quantiles), np.nan)

    # 排序
    sorter = np.argsort(values)
    values = values[sorter]
    weights = weights[sorter]

    # 累积权重
    cdf = np.cumsum(weights)
    cdf = cdf / cdf[-1]

    # 插值获取分位数
    return np.interp(quantiles, cdf, values)


def calc_area_weights(lat):
    """
    根据纬度计算面积权重：cos(lat)
    """
    return np.cos(np.deg2rad(lat))


def calc_weighted_mean_std(data, lat_name="lat", lon_name="lon"):
    """
    计算面积加权均值和标准差
    """
    weights = calc_area_weights(data[lat_name])

    mean_w = data.weighted(weights).mean(dim=(lat_name, lon_name), skipna=True)
    var_w = ((data - mean_w) ** 2).weighted(weights).mean(dim=(lat_name, lon_name), skipna=True)
    std_w = np.sqrt(var_w)

    return mean_w, std_w


def calc_unweighted_quantiles(data, quantiles, lat_name="lat", lon_name="lon"):
    """
    计算不加权分位数
    """
    q = data.quantile(quantiles, dim=(lat_name, lon_name), skipna=True)
    return q


def calc_weighted_quantiles(data, quantiles, lat_name="lat", lon_name="lon"):
    """
    计算面积加权分位数
    适用于二维(lat, lon)数据，或额外带有其它维度（如 time）
    """
    weights_1d = calc_area_weights(data[lat_name])
    weights_2d = weights_1d.broadcast_like(data)

    def _wq(values, weights):
        return weighted_quantile_1d(values, weights, quantiles)

    q = xr.apply_ufunc(
        _wq,
        data,
        weights_2d,
        input_core_dims=[[lat_name, lon_name], [lat_name, lon_name]],
        output_core_dims=[["quantile"]],
        vectorize=True,
        dask="parallelized",
        output_dtypes=[float],
        dask_gufunc_kwargs={"output_sizes": {"quantile": len(np.atleast_1d(quantiles))}},
    )

    q = q.assign_coords(quantile=np.atleast_1d(quantiles))
    return q


def analyze_spatial_data(
    file_path,
    var_name,
    quantiles=(0.1, 0.25, 0.5, 0.75, 0.9),
    lat_name="lat",
    lon_name="lon",
):
    """
    读取数据并计算：
    - 面积加权均值
    - 面积加权标准差
    - 不加权分位数
    - 面积加权分位数
    """
    ds = xr.open_dataset(file_path)
    data = ds[var_name]

    print("变量维度:", data.dims)

    mean_w, std_w = calc_weighted_mean_std(data, lat_name=lat_name, lon_name=lon_name)
    q_unw = calc_unweighted_quantiles(data, quantiles, lat_name=lat_name, lon_name=lon_name)
    q_w = calc_weighted_quantiles(data, quantiles, lat_name=lat_name, lon_name=lon_name)

    return {
        "dataset": ds,
        "data": data,
        "weighted_mean": mean_w,
        "weighted_std": std_w,
        "quantiles_unweighted": q_unw,
        "quantiles_weighted": q_w,
    }



In [2]:

if __name__ == "__main__":
    file_path = r"F:\User_file\wyy\SPDB\part3_forecast\sw\output_2020\sw_value_year_mean.nc"
    var_name = "sw_year_mean"

    result = analyze_spatial_data(
        file_path=file_path,
        var_name=var_name,
        quantiles=(0.1, 0.25, 0.5, 0.75, 0.9),
    )

    print(f"面积加权均值: {float(result['weighted_mean'].values):.6f}")
    print(f"面积加权标准差: {float(result['weighted_std'].values):.6f}")

    print("\n不加权分位数:")
    print(result["quantiles_unweighted"])

    print("\n面积加权分位数:")
    print(result["quantiles_weighted"])

变量维度: ('lat', 'lon')
面积加权均值: 8.078853
面积加权标准差: 51.692626

不加权分位数:
<xarray.DataArray 'sw_year_mean' (quantile: 5)>
array([ 1.11577455,  1.55606256,  3.48826533,  6.18669406, 12.86481774])
Coordinates:
  * quantile  (quantile) float64 0.1 0.25 0.5 0.75 0.9

面积加权分位数:
<xarray.DataArray (quantile: 5)>
array([ 1.13443442,  1.72233362,  3.56342373,  6.81024837, 13.80760711])
Coordinates:
  * quantile  (quantile) float64 0.1 0.25 0.5 0.75 0.9


In [3]:

if __name__ == "__main__":
    file_path = r"F:\User_file\wyy\SPDB\part3_forecast\lr\output_2020\lr_value_year_mean.nc"
    var_name = "lr_year_mean"

    result = analyze_spatial_data(
        file_path=file_path,
        var_name=var_name,
        quantiles=(0.1, 0.25, 0.5, 0.75, 0.9),
    )

    print(f"面积加权均值: {float(result['weighted_mean'].values):.6f}")
    print(f"面积加权标准差: {float(result['weighted_std'].values):.6f}")

    print("\n不加权分位数:")
    print(result["quantiles_unweighted"])

    print("\n面积加权分位数:")
    print(result["quantiles_weighted"])

变量维度: ('lat', 'lon')
面积加权均值: 3.496448
面积加权标准差: 2.147053

不加权分位数:
<xarray.DataArray 'lr_year_mean' (quantile: 5)>
array([2.21373758, 2.64011217, 2.98637508, 3.63391556, 5.37064973])
Coordinates:
  * quantile  (quantile) float64 0.1 0.25 0.5 0.75 0.9

面积加权分位数:
<xarray.DataArray (quantile: 5)>
array([2.10594002, 2.5337632 , 2.91198542, 3.50761558, 5.35739166])
Coordinates:
  * quantile  (quantile) float64 0.1 0.25 0.5 0.75 0.9
